# L5 — Agente autónomo

> **Le das un objetivo, no instrucciones.** El agente decide por sí solo qué herramientas usar, en qué orden, y cuándo ha terminado.

## ¿Qué vas a aprender?

- La diferencia entre **tool use** (L3) y un **agente** real
- Qué es el **bucle agente**: observar → pensar → actuar → evaluar
- Por qué el system prompt define un **protocolo de trabajo**, no solo un rol
- Cómo poner un **límite de iteraciones** para evitar bucles infinitos

## ¿Qué es un agente?

Un agente es un modelo al que le das un **objetivo abierto** y deja de esperar instrucciones paso a paso.

En los niveles anteriores siempre había un humano (o el código) controlando el flujo:
- **L2**: el programador define exactamente los pasos y su orden
- **L3**: el modelo elige qué tools invocar, pero responde a una pregunta concreta y puntual

En **L5** el modelo recibe un objetivo abierto — *"investiga este incidente y resuélvelo"* — y trabaja de forma autónoma hasta completarlo.

## El bucle agente

```
┌─────────────────────────────────────────┐
│   Objetivo                              │
│      ↓                                  │
│   Observar  →  ¿qué sé hasta ahora?     │
│      ↓                                  │
│   Pensar    →  ¿qué debo hacer ahora?   │
│      ↓                                  │
│   Actuar    →  invocar tool             │
│      ↓                                  │
│   Evaluar   →  ¿he completado el obj.?  │
│      ↓ no                               │
│   (volver a Observar)                   │
│      ↓ sí                               │
│   Terminar  →  informe final            │
└─────────────────────────────────────────┘
```

**La diferencia clave con L3** está en el **Evaluar**: en L3 el bucle termina cuando el modelo no necesita más tools. En L5 el bucle termina cuando el modelo decide que ha cumplido el objetivo — son condiciones de parada distintas.

## Por qué es L5 y no L3

En L3 la pregunta era puntual: *"clasifica este ticket"*. El modelo usaba dos tools, producía un JSON y terminaba.

En L5 el objetivo es **compuesto y abierto**:

> *"Investiga este incidente. Determina la causa. Crea el ticket con la severidad correcta. Si es P1 o P2, escala al equipo correspondiente. Redacta el informe final."*

El modelo tiene que:
1. Decidir qué investigar primero basándose en el reporte inicial
2. **Adaptar los siguientes pasos según lo que encuentre**
3. Tomar acciones con efecto real (crear ticket, escalar)
4. Decidir activamente que el objetivo está cumplido

## Conceptos clave

### Goal-directed behavior
El agente trabaja hacia un **objetivo**, no hacia una respuesta. Una respuesta termina cuando el modelo dice algo. Un objetivo termina cuando el modelo *hace* algo — crear un ticket, escalar un incidente, notificar a un equipo.

### El system prompt como definición del agente
En L3 el system prompt describía el rol. En L5 describe el **protocolo de trabajo**: qué fases seguir, qué criterios usar para decidir, qué constituye "terminado". Es la diferencia entre *"eres un ingeniero de soporte"* y *"eres un ingeniero de soporte y este es tu proceso"*.

### Autonomía y sus riesgos
Más autonomía = más capacidad **y más superficie de error**. Un agente puede:
- Encadenarse en un bucle si el objetivo no está bien definido
- Tomar decisiones incorrectas si las tools devuelven datos ambiguos
- Escalar innecesariamente si los criterios son vagos

Por eso el system prompt de un agente real **invierte mucho más cuidado** que el de una chain.

### Límite de iteraciones
En producción siempre hay un techo al bucle — si el agente lleva N rondas sin terminar, algo ha ido mal. Aquí: `MAX_ITERATIONS = 10`.

> **Requisitos:** `pip install anthropic python-dotenv`

---
## Setup

Cargamos el cliente y definimos el helper de parsing JSON. **Ojo a `MAX_ITERATIONS = 10`** — es la red de seguridad.

In [1]:
"""
L5 — Agente autónomo de respuesta a incidentes
===============================================
El agente recibe un reporte en texto libre y trabaja de forma autónoma
hasta completar su objetivo: investigar, diagnosticar, actuar e informar.

A diferencia de L3, aquí el modelo no responde a una pregunta — trabaja
hacia un objetivo compuesto que él mismo decide cuándo ha cumplido.

El bucle es el mismo que en L3 (stop_reason == "tool_use"), pero la
condición de parada es distinta: el modelo termina cuando considera
que el objetivo está completo, no solo cuando ya no necesita tools.

Requisitos:
    pip install anthropic

Variables de entorno:
    ANTHROPIC_API_KEY
"""

from pathlib import Path
from dotenv import load_dotenv
for _p in [Path.cwd()] + list(Path.cwd().parents):
    if (_p / ".env").exists():
        load_dotenv(_p / ".env", override=True)
        break
import json
import anthropic

client = anthropic.Anthropic()
MODEL          = "claude-sonnet-4-5"
MAX_ITERATIONS = 10  # techo de seguridad — si se alcanza, algo ha ido mal

def parse_json_safe(text: str) -> dict:
    import re
    # Busca JSON dentro de ```json ... ``` o ``` ... ```
    match = re.search(r"```(?:json)?\s*([\s\S]*?)```", text)
    if match:
        return json.loads(match.group(1).strip())
    # Si no hay backticks, busca directamente el primer {
    match = re.search(r"\{[\s\S]*\}", text)
    if match:
        return json.loads(match.group(0))
    return json.loads(text.strip())

## Definición de tools

El agente tiene **6 tools** disponibles. Fíjate en la mezcla:
- **Tools de observación** (read-only): `check_service_health`, `get_incident_history`, `search_runbook`
- **Tools de acción** (con efecto real): `create_incident`, `escalate_incident`, `add_incident_note`

Las descripciones le dicen al modelo **cuándo** usar cada una — incluso reglas como *"siempre llama a esta tool al inicio"* o *"obligatorio para P1 y P2"*.

In [2]:
# ─────────────────────────────────────────────
# Definición de tools
# ─────────────────────────────────────────────

TOOLS = [
    {
        "name": "check_service_health",
        "description": (
            "Consulta el estado actual de un servicio en producción. "
            "Devuelve métricas en tiempo real: tasa de error, latencia, "
            "y si hay un incidente activo. Úsala siempre al inicio para "
            "establecer el estado base antes de investigar más."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "service": {
                    "type": "string",
                    "description": "Nombre del servicio: auth, payments, api_gateway, database, frontend"
                }
            },
            "required": ["service"]
        }
    },
    {
        "name": "get_incident_history",
        "description": (
            "Recupera los incidentes más recientes de un servicio. "
            "Útil para detectar patrones — si el mismo servicio ha fallado "
            "tres veces esta semana, eso cambia la severidad y la acción."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "service": {
                    "type": "string",
                    "description": "Nombre del servicio a consultar"
                },
                "limit": {
                    "type": "integer",
                    "description": "Número de incidentes a devolver (default: 3)"
                }
            },
            "required": ["service"]
        }
    },
    {
        "name": "search_runbook",
        "description": (
            "Busca en los runbooks y guías de operaciones el procedimiento "
            "de resolución para un tipo de problema. Devuelve los pasos "
            "recomendados y el equipo responsable."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "Descripción del problema a buscar, ej: 'auth 500 database connection'"
                }
            },
            "required": ["query"]
        }
    },
    {
        "name": "create_incident",
        "description": (
            "Crea un incidente en el sistema de gestión. "
            "Llama a esta tool SIEMPRE, incluso para incidentes menores — "
            "todo debe quedar registrado. Devuelve el ID del incidente creado."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "title":       {"type": "string", "description": "Título conciso del incidente"},
                "description": {"type": "string", "description": "Descripción técnica detallada"},
                "severity":    {"type": "string", "description": "P1 | P2 | P3 | P4"},
                "service":     {"type": "string", "description": "Servicio afectado"}
            },
            "required": ["title", "description", "severity", "service"]
        }
    },
    {
        "name": "escalate_incident",
        "description": (
            "Escala un incidente al equipo responsable. "
            "Obligatorio para P1 y P2. Opcional para P3 si hay reincidencia. "
            "Nunca escalar P4 — se gestiona en el backlog normal."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "incident_id": {"type": "string", "description": "ID del incidente a escalar"},
                "team": {
                    "type": "string",
                    "description": "Equipo al que escalar: backend, frontend, infra, database, security"
                },
                "reason": {"type": "string", "description": "Razón de la escalada"}
            },
            "required": ["incident_id", "team", "reason"]
        }
    },
    {
        "name": "add_incident_note",
        "description": (
            "Añade una nota de investigación a un incidente existente. "
            "Usar para documentar hallazgos intermedios — qué se investigó, "
            "qué se descartó y por qué."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "incident_id": {"type": "string"},
                "note":        {"type": "string", "description": "Hallazgo o acción documentada"}
            },
            "required": ["incident_id", "note"]
        }
    }
]

## Implementación de las tools (mocks)

En producción cada función llamaría a tu sistema real (Kubernetes, PagerDuty, Jira...). Aquí devuelven datos simulados pero **realistas** — datos suficientemente ricos para que el agente tome decisiones interesantes.

`execute_tool` despacha al implementador correcto y devuelve el resultado como JSON string (que es lo que la API de Anthropic espera para `tool_result`).

In [3]:
# ─────────────────────────────────────────────
# Implementación de tools (mocks)
# ─────────────────────────────────────────────

_incident_counter = 100  # simula un ID autoincremental


def check_service_health(service: str) -> dict:
    statuses = {
        "auth": {
            "status": "degraded",
            "error_rate_pct": 43,
            "p99_latency_ms": 8200,
            "active_incident": True,
            "notes": "Spike in connection errors since 09:12 UTC"
        },
        "payments": {
            "status": "operational",
            "error_rate_pct": 0.1,
            "p99_latency_ms": 320,
            "active_incident": False
        },
        "api_gateway": {
            "status": "operational",
            "error_rate_pct": 0.3,
            "p99_latency_ms": 95,
            "active_incident": False
        },
        "database": {
            "status": "degraded",
            "error_rate_pct": 12,
            "p99_latency_ms": 4100,
            "active_incident": True,
            "notes": "Connection pool at 98% capacity. Max connections: 200"
        },
        "frontend": {
            "status": "operational",
            "error_rate_pct": 0.0,
            "p99_latency_ms": 210,
            "active_incident": False
        },
    }
    return statuses.get(service, {"status": "unknown", "error": f"Service '{service}' not found"})


def get_incident_history(service: str, limit: int = 3) -> list[dict]:
    history = {
        "auth": [
            {
                "id": "INC-089",
                "date": "2024-01-08",
                "title": "Auth 500s due to DB connection pool exhaustion",
                "resolution": "Increased max_connections to 200, restarted auth service",
                "duration_min": 23,
                "severity": "P1"
            },
            {
                "id": "INC-071",
                "date": "2023-12-19",
                "title": "Auth failures after deploy — bad env var",
                "resolution": "Rolled back deploy, fixed DATABASE_URL in config",
                "duration_min": 45,
                "severity": "P1"
            },
        ],
        "database": [
            {
                "id": "INC-082",
                "date": "2024-01-05",
                "title": "DB connection pool exhausted during peak traffic",
                "resolution": "Killed idle-in-transaction connections, added connection pooling via PgBouncer",
                "duration_min": 35,
                "severity": "P2"
            }
        ],
        "api_gateway": [
            {
                "id": "INC-061",
                "date": "2023-11-30",
                "title": "Circuit breaker open on payments upstream",
                "resolution": "Restarted Kong, payments service had recovered",
                "duration_min": 12,
                "severity": "P2"
            }
        ],
    }
    return history.get(service, [])[:limit]


def search_runbook(query: str) -> dict:
    query_lower = query.lower()

    if any(k in query_lower for k in ["auth", "login", "500", "connection"]):
        return {
            "runbook": "Auth Service — Connection Errors",
            "steps": [
                "1. Check DB connection pool: SELECT count(*) FROM pg_stat_activity WHERE state='active'",
                "2. If pool > 180: kill idle-in-transaction connections older than 10min",
                "3. Restart auth service: kubectl rollout restart deployment/auth-service",
                "4. Monitor for 2 minutes: kubectl logs -f deployment/auth-service",
                "5. Verify /auth/health returns 200"
            ],
            "owner_team": "backend",
            "escalate_if": "Pool exhausted repeatedly or restart doesn't resolve within 5 min"
        }

    if any(k in query_lower for k in ["database", "db", "pool", "timeout"]):
        return {
            "runbook": "Database — Connection Pool Issues",
            "steps": [
                "1. Query active connections by service: SELECT application_name, count(*) FROM pg_stat_activity GROUP BY 1",
                "2. Identify the culprit service and check for connection leaks",
                "3. Kill long-running idle transactions: SELECT pg_terminate_backend(pid) ...",
                "4. Consider enabling PgBouncer if issue is recurring"
            ],
            "owner_team": "database",
            "escalate_if": "Connections don't drop after killing idle transactions"
        }

    return {
        "runbook": "General Incident Response",
        "steps": [
            "1. Identify affected service using monitoring dashboards",
            "2. Check recent deploys: any correlation with incident start time?",
            "3. Review logs for the affected service",
            "4. Escalate to the service owner if no resolution in 15 min"
        ],
        "owner_team": "on-call",
        "escalate_if": "No root cause found within 20 minutes"
    }


def create_incident(title: str, description: str, severity: str, service: str) -> dict:
    global _incident_counter
    _incident_counter += 1
    incident_id = f"INC-{_incident_counter}"
    sla = {"P1": "1h", "P2": "4h", "P3": "24h", "P4": "72h"}.get(severity, "24h")
    return {
        "id":        incident_id,
        "title":     title,
        "severity":  severity,
        "service":   service,
        "status":    "open",
        "sla":       sla,
        "created":   "2024-01-15T09:18:00Z",
    }


def escalate_incident(incident_id: str, team: str, reason: str) -> dict:
    return {
        "incident_id":  incident_id,
        "escalated_to": team,
        "channel":      f"#incidents-{team}",
        "notified":     True,
        "message":      f"[{incident_id}] Escalated to {team}: {reason}"
    }


def add_incident_note(incident_id: str, note: str) -> dict:
    return {"incident_id": incident_id, "note_saved": True, "note": note}


def execute_tool(name: str, inputs: dict) -> str:
    """
    Punto de control: el modelo solicita, el código ejecuta.
    En producción aquí van validaciones, rate limiting, audit log.
    """
    print(f"  [Tool] {name}({json.dumps(inputs, ensure_ascii=False)})")

    dispatch = {
        "check_service_health":  lambda: check_service_health(**inputs),
        "get_incident_history":  lambda: get_incident_history(**inputs),
        "search_runbook":        lambda: search_runbook(**inputs),
        "create_incident":       lambda: create_incident(**inputs),
        "escalate_incident":     lambda: escalate_incident(**inputs),
        "add_incident_note":     lambda: add_incident_note(**inputs),
    }

    fn = dispatch.get(name)
    if fn is None:
        return json.dumps({"error": f"Tool '{name}' no encontrada"})

    return json.dumps(fn(), ensure_ascii=False, indent=2)

## El system prompt — define el agente

Este es el corazón del agente. Fíjate en la estructura:
1. **Identidad**: *"You are an autonomous incident response agent"*
2. **Objetivo en 4 fases**: INVESTIGATE → DIAGNOSE → ACT → REPORT
3. **Criterios explícitos** para tomar decisiones (P1/P2/P3/P4)
4. **Reglas obligatorias**: cuándo llamar cada tool
5. **Schema del output final**: cuándo y cómo terminar

Sin este nivel de detalle, el agente sería errático.

In [4]:
# ─────────────────────────────────────────────
# System prompt — define el agente
#
# En L3 el prompt describía un rol.
# En L5 describe un protocolo de trabajo completo:
# fases, criterios de decisión y condición de parada.
# ─────────────────────────────────────────────

SYSTEM_PROMPT = """
You are an autonomous incident response agent. When you receive an incident report,
you work through it independently until the objective is fully complete.

Your objective for every incident:
  1. INVESTIGATE — gather facts using the available tools
  2. DIAGNOSE    — determine root cause and severity
  3. ACT         — create the incident ticket; escalate if P1 or P2
  4. REPORT      — produce the final JSON report

Severity criteria:
  P1: production system down or data loss — immediate escalation required
  P2: critical functionality degraded in production — escalate within 15 min
  P3: bug in production with no data loss, workaround exists — no escalation
  P4: minor issue or question — no escalation, handle in backlog

Rules:
  - Always call check_service_health before drawing conclusions
  - Always call get_incident_history to check for patterns (recurring = higher severity)
  - Always call search_runbook to find the resolution procedure
  - Always call create_incident — every incident must be recorded, even P4
  - Call escalate_incident for P1 and P2 without exception
  - Use add_incident_note to document key findings during investigation

When you have completed all four phases, respond with ONLY valid JSON:
{
  "incident_id":    string,
  "root_cause":     string,
  "severity":       "P1" | "P2" | "P3" | "P4",
  "actions_taken":  [string],
  "escalated":      boolean,
  "escalated_to":   string | null,
  "resolution":     string,
  "next_steps":     [string]
}
""".strip()

## Bucle del agente — `run_agent`

Técnicamente el bucle es **igual que en L3**: while → llamar al modelo → si `tool_use` ejecutar y continuar, si `end_turn` parsear y devolver.

**La diferencia conceptual:** el modelo llega a `end_turn` no porque "no necesite más tools", sino porque ha completado activamente las cuatro fases del objetivo.

In [5]:
# ─────────────────────────────────────────────
# Bucle del agente
# ─────────────────────────────────────────────

def run_agent(report: str) -> dict:
    """
    Ejecuta el agente hasta que completa el objetivo o alcanza MAX_ITERATIONS.

    El bucle es técnicamente igual que en L3, pero el modelo llega a
    end_turn de forma distinta: no porque se haya quedado sin tools que
    llamar, sino porque ha completado las cuatro fases del objetivo y
    decide activamente que ha terminado.
    """
    print("=" * 60)
    print("INCIDENTE:", report[:80] + "..." if len(report) > 80 else report)
    print("=" * 60)

    messages   = [{"role": "user", "content": f"[INCIDENT REPORT]\n{report}\n[/INCIDENT REPORT]"}]
    iterations = 0

    while iterations < MAX_ITERATIONS:
        iterations += 1
        print(f"\n[Iteración {iterations}]")

        response = client.messages.create(
            model=MODEL,
            max_tokens=2048,
            temperature=0,
            system=SYSTEM_PROMPT,
            tools=TOOLS,
            messages=messages,
        )

        print(f"  stop_reason: {response.stop_reason}")

        # El agente ha decidido que el objetivo está completo
        if response.stop_reason == "end_turn":
            text = next((b.text for b in response.content if b.type == "text"), "")
            print("DEBUG raw:", repr(text))  # añade esto
            result = parse_json_safe(text)
            print("\n" + "=" * 60)
            print("INFORME FINAL:")
            print(json.dumps(result, ensure_ascii=False, indent=2))
            return result

        # El agente quiere ejecutar tools — seguimos en el bucle
        if response.stop_reason == "tool_use":
            messages.append({"role": "assistant", "content": response.content})

            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    result = execute_tool(block.name, block.input)
                    tool_results.append({
                        "type":        "tool_result",
                        "tool_use_id": block.id,
                        "content":     result,
                    })

            messages.append({"role": "user", "content": tool_results})

    # Límite de iteraciones alcanzado — el agente no terminó
    print(f"\n[WARNING] Límite de {MAX_ITERATIONS} iteraciones alcanzado.")
    return {"error": "max_iterations_reached", "iterations": iterations}

## Escenarios de ejemplo

Tres incidentes de complejidad creciente:
- **`incident_critical`** → P1 claro: caída masiva, debería escalar
- **`incident_minor`** → P3/P4: bug visual, no debería escalar
- **`incident_ambiguous`** → ambiguo: el agente tiene que **investigar** para determinar el alcance

In [6]:
# ─────────────────────────────────────────────
# Escenarios de ejemplo
# ─────────────────────────────────────────────

# P1: servicio caído, impacto masivo
incident_critical = """
Users cannot log in since 09:12 UTC. The authentication service is returning
500 errors on POST /auth/login. Approximately 3,500 users impacted across all regions.
Database connection timeouts visible in the logs. The issue started suddenly,
no deploy was made in the last 6 hours.
""".strip()

# P3: bug menor, sin impacto en datos
incident_minor = """
On the account settings page, when a user updates their email address,
the confirmation toast shows the old email instead of the new one.
The update itself saves correctly — it is purely a display issue.
Reported by 2 users in production. Low impact.
""".strip()

# Ambiguo: el agente tiene que investigar para determinar el alcance
incident_ambiguous = """
We're getting some complaints from users about slow response times on the dashboard.
Not sure which service is the problem. Started maybe 30 minutes ago.
Some users say it times out, others say it just feels slow.
""".strip()

## Ejecutar — incidente crítico

Observa cómo el agente:
1. Llama a `check_service_health` para establecer estado base
2. Consulta `get_incident_history` para ver patrones
3. Busca un `search_runbook` con el procedimiento
4. Crea el ticket con `create_incident`
5. Escala con `escalate_incident`
6. Devuelve el informe final

In [7]:
run_agent(incident_critical)

INCIDENTE: Users cannot log in since 09:12 UTC. The authentication service is returning
500...

[Iteración 1]
  stop_reason: tool_use
  [Tool] check_service_health({"service": "auth"})
  [Tool] check_service_health({"service": "database"})
  [Tool] get_incident_history({"service": "auth", "limit": 5})
  [Tool] search_runbook({"query": "auth 500 database connection timeout login"})

[Iteración 2]
  stop_reason: tool_use
  [Tool] create_incident({"title": "Auth service down - 500 errors on login due to database connection pool exhaustion", "description": "Authentication service returning 500 errors on POST /auth/login since 09:12 UTC. Approximately 3,500 users impacted across all regions. Database connection timeouts visible in logs. Auth service health: 43% error rate, 8200ms p99 latency. Database connection pool at 98% capacity (196/200 connections). No recent deploys. Pattern match: Similar to INC-089 (2024-01-08) - DB connection pool exhaustion. Runbook: Auth Service Connection Error

{'incident_id': 'INC-101',
 'root_cause': 'Database connection pool exhaustion (98% capacity, 196/200 connections) preventing auth service from acquiring connections to process login requests. No connection leaks from idle-in-transaction queries causing pool starvation.',
 'severity': 'P1',
 'actions_taken': ['Checked auth service health: 43% error rate, 8200ms p99 latency, degraded status',
  'Checked database health: 12% error rate, 4100ms p99 latency, connection pool at 98% capacity',
  'Retrieved incident history: identified recurring pattern (INC-089 on 2024-01-08, INC-071 on 2023-12-19)',
  'Located runbook procedure: Auth Service Connection Errors',
  'Created incident INC-101 with P1 severity',
  'Escalated to backend team via #incidents-backend channel',
  'Documented investigation findings and recommended immediate actions'],
 'escalated': True,
 'escalated_to': 'backend',
 'resolution': 'Immediate: Backend team notified to execute runbook procedure - kill idle DB connections

Ahora el incidente menor — fíjate en cómo cambia la cantidad de tools invocadas y la severidad asignada:

In [8]:
run_agent(incident_minor)

INCIDENTE: On the account settings page, when a user updates their email address,
the confi...

[Iteración 1]
  stop_reason: tool_use
  [Tool] check_service_health({"service": "frontend"})
  [Tool] get_incident_history({"service": "frontend", "limit": 3})
  [Tool] search_runbook({"query": "frontend display issue toast confirmation email update"})

[Iteración 2]
  stop_reason: tool_use
  [Tool] create_incident({"title": "Account settings: Email update confirmation toast displays old email instead of new email", "description": "Display bug on account settings page. When users update their email address, the confirmation toast message shows the old email address instead of the newly updated email. The actual email update functionality works correctly and saves the new email to the database - this is purely a frontend display issue in the toast notification component. Reported by 2 users in production. No data integrity issues. Frontend service is operational with 0% error rate.", "severit

{'incident_id': 'INC-102',
 'root_cause': 'Frontend toast notification component displaying stale email value instead of updated email after account settings update. The toast is likely reading from cached/stale state rather than the newly updated value returned from the API call.',
 'severity': 'P4',
 'actions_taken': ['Checked frontend service health - confirmed operational (0% error rate, 210ms p99 latency)',
  'Reviewed incident history - no similar or recurring issues found',
  'Searched runbook for resolution procedures',
  'Created incident ticket INC-102 with P4 severity',
  'Documented investigation findings in incident notes'],
 'escalated': False,
 'escalated_to': None,
 'resolution': 'P4 minor UI bug - will be handled in normal development backlog. The issue is isolated to the toast notification display logic and does not affect core functionality. Users can verify their email update by refreshing the page or navigating away and back to settings.',
 'next_steps': ['Frontend

Y el ambiguo — aquí el agente tiene que decidir por sí mismo qué investigar:

In [9]:
run_agent(incident_ambiguous)

INCIDENTE: We're getting some complaints from users about slow response times on the dashbo...

[Iteración 1]
  stop_reason: tool_use
  [Tool] check_service_health({"service": "frontend"})
  [Tool] check_service_health({"service": "api_gateway"})
  [Tool] check_service_health({"service": "database"})
  [Tool] check_service_health({"service": "auth"})

[Iteración 2]
  stop_reason: tool_use
  [Tool] get_incident_history({"service": "database", "limit": 5})
  [Tool] get_incident_history({"service": "auth", "limit": 5})
  [Tool] search_runbook({"query": "database connection pool capacity high latency timeout"})

[Iteración 3]
  stop_reason: tool_use
  [Tool] create_incident({"title": "Dashboard slow/timeout - DB connection pool exhaustion causing auth failures", "description": "Users reporting slow dashboard response times and timeouts starting ~30 min ago. Investigation reveals:\n- Database connection pool at 98% capacity (196/200 connections)\n- Auth service degraded: 43% error rate, 820

{'incident_id': 'INC-103',
 'root_cause': 'Database connection pool exhaustion (98% capacity, 196/200 connections) causing cascading failure to auth service. Auth service experiencing 43% error rate and 8200ms p99 latency, resulting in dashboard timeouts and slow response times for users. This is the 3rd occurrence in 2 weeks despite previous mitigations (PgBouncer, increased max_connections), indicating a persistent connection leak or inadequate pooling strategy.',
 'severity': 'P2',
 'actions_taken': ['Checked health status of frontend, api_gateway, database, and auth services',
  'Identified database connection pool at 98% capacity with 4100ms p99 latency and 12% error rate',
  'Identified auth service degraded with 43% error rate and 8200ms p99 latency',
  'Reviewed incident history revealing recurring pattern (INC-089 on Jan 8, INC-082 on Jan 5)',
  'Retrieved runbook for database connection pool exhaustion',
  'Created incident INC-103 with P2 severity',
  'Escalated to backend t

## Siguientes pasos → L6

Un agente solo resuelve problemas en serie — uno detrás de otro. Cuando los problemas son **complejos y paralelizables**, la solución es **multi-agente**: varios agentes especializados trabajando en paralelo, coordinados por un orquestador.

→ **[L6 — Multi-agente](../L6-MULTI_AGENT/L6_Multi_Agent.ipynb)**